<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI2027_Phase30_Lineage_Athena_Gate_Drive_to_Drive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICHI 2027 — Phase 30
## Deterministic lineage reconciliation + optional private Athena concept review
**Drive → Drive • Fully synthetic fixture lineage • No invented OMOP concept IDs**

**Inputs required from Phase 29 (same Drive folder):** `ICHI2027_Phase29_SHAREABLE.json`, `ICHI2027_Phase29_SYNTHETIC_FHIR_WITH_PROCEDURES.json`, `ICHI2027_Phase29_SYNTHETIC_LINEAGE_SIDECAR.csv`, `ICHI2027_Phase29_GAP_DISPOSITION.csv`, `ICHI2027_Phase29_BINDING_AUDIT.csv`, `ICHI2027_Phase29_TARGET_REQUIRED_FIELD_AUDIT.csv`. The notebook checks Phase 29's exact bundle SHA-256 and recorded gate status.

**New work:** independently reconstruct the source-to-target mapping for all 16 synthetic Observations; verify provenance, laterality, time points, numeric units, person/study/procedure links, and stable feature IDs; run negative controls. If the original Phase 27 staging and provenance sidecar are still in this folder, compare against them **independently**; otherwise report `NOT_ATTESTED` instead of assuming row-order equivalence. Produce a 19-key human-review sheet. Optionally validate human-reviewed concepts against one **private, frozen** Athena `CONCEPT.csv` snapshot.

**Not done by this notebook:** automatic selection or approval of concept IDs; real ADNI/DICOM provenance; pyOMOP execution; official MI-CDM loading; Arm A completion; Arm B/C/D comparison; OMOP-target cohort-equivalence testing. A valid synthetic lineage test does not establish those results.

**To validate vocabulary privately (optional):** place the Athena export at `<the Phase29 folder>/_PRIVATE_ATHENA/CONCEPT.csv` (or `.csv.gz`). This should be the complete, single, licensed snapshot you downloaded. First run creates `ICHI2027_Phase30_BINDING_REVIEW_TEMPLATE.csv` in the Phase29 folder. Copy it to `ICHI2027_Phase30_REVIEWED_BINDINGS.csv`, fill only the choices you have independently reviewed, and run again. The snapshot and private candidate suggestions are never copied to public output files. Without them the lineage work completes, but concept gates stay blocked.

In [1]:
# 1 — Mount Drive. For local regression testing only, ICHI2027_DRIVE_ROOT overrides MyDrive.
import os, json, hashlib, copy, re, gzip, csv
from pathlib import Path
from datetime import datetime, timezone, date
import pandas as pd

if os.environ.get('ICHI2027_DRIVE_ROOT'):
    DRIVE_ROOT=Path(os.environ['ICHI2027_DRIVE_ROOT'])
else:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT=Path('/content/drive/MyDrive')

# Optional: set a precise folder relative to MyDrive, if you have multiple Phase29 runs.
DRIVE_RELATIVE_FOLDER=''
ROOT=DRIVE_ROOT / DRIVE_RELATIVE_FOLDER if DRIVE_RELATIVE_FOLDER else DRIVE_ROOT
if not ROOT.is_dir():
    raise FileNotFoundError(f'Google Drive folder is not accessible: {ROOT}')
print('Search root:',ROOT)

Mounted at /content/drive
Search root: /content/drive/MyDrive


In [2]:
# 2 — Robust matching: safely ignore stale Google Drive entries, check paired Phase29 bundle SHA.
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()

def available(root, pattern):
    results=[]
    for folder, dirs, names in os.walk(root,onerror=lambda _:None):
        dirs[:]=[d for d in dirs if not d.startswith('.') and d not in {'node_modules','.git','_PRIVATE_ATHENA'}]
        for name in names:
            if Path(name).match(pattern):
                p=Path(folder)/name
                try:
                    if p.is_file():results.append(p)
                except (FileNotFoundError,OSError):pass
    def mtime(p):
        try:return p.stat().st_mtime
        except (OSError,FileNotFoundError):return -1
    return sorted(results,key=mtime,reverse=True)

def sibling(folder,pattern,expected_sha=None):
    for p in available(folder,pattern):
        if p.parent!=folder:continue
        try:
            if expected_sha is None or sha256(p)==expected_sha:return p
        except (OSError,FileNotFoundError):continue
    return None

PAIRED={
    'bundle':'ICHI2027_Phase29_SYNTHETIC_FHIR_WITH_PROCEDURES*.json',
    'lineage':'ICHI2027_Phase29_SYNTHETIC_LINEAGE_SIDECAR*.csv',
    'gaps':'ICHI2027_Phase29_GAP_DISPOSITION*.csv',
    'bindings':'ICHI2027_Phase29_BINDING_AUDIT*.csv',
    'required':'ICHI2027_Phase29_TARGET_REQUIRED_FIELD_AUDIT*.csv'
}
selected=None;rejected=[]
for jp in available(ROOT,'ICHI2027_Phase29_SHAREABLE*.json'):
    try:
        s=json.loads(jp.read_text(encoding='utf-8'))
        if s.get('phase')!='29_synthetic_procedure_fixture_and_frozen_vocabulary_gate':continue
        if (s.get('synthetic_fixture',{}).get('tests_passing')!=6 or
            not s.get('gates',{}).get('exact_phase27_imagewg_inventory_read_back')):
            rejected.append((jp.name,'Phase29 integrity gates not met'));continue
        folder=jp.parent
        files={k:sibling(folder,pattern,s['synthetic_fixture']['patched_bundle_sha256'] if k=='bundle' else None)
               for k,pattern in PAIRED.items()}
        if all(files.values()):
            selected=(jp,s,files);break
        rejected.append((jp.name,'Missing companion file or bundle SHA mismatch'))
    except (FileNotFoundError,OSError,ValueError,KeyError) as exc:
        rejected.append((jp.name,type(exc).__name__))
if not selected:
    raise FileNotFoundError('No valid paired Phase 29 run found under Drive. '+str(rejected[:5]))
P29_JSON,p29,paths=selected
OUT_DIR=P29_JSON.parent
bundle=json.loads(paths['bundle'].read_text(encoding='utf-8'))
lineage29=pd.read_csv(paths['lineage'],dtype={'source_observation_id':str},keep_default_na=False)
gaps29=pd.read_csv(paths['gaps'],dtype={'staging_row_id':str},keep_default_na=False)
bind29=pd.read_csv(paths['bindings'],dtype={'binding_key':str},keep_default_na=False)
required29=pd.read_csv(paths['required'],keep_default_na=False)

assert len(lineage29)==16 and len(gaps29)==48 and len(bind29)==19
assert (gaps29.resolution_status=='BLOCKED_VOCABULARY_VALIDATION').sum()==36
assert p29['original_48_gap_disposition']['remaining_blocked']==36
assert p29['gates']['synthetic_faithful_armA_mapping_completed'] is False
assert p29['gates']['real_MRI_rows_processed']==0
assert p29['pinned_imagewg_commit']=='8f0a19bb94e941bb335a097bc9851fab672621c7'
assert p29['pinned_omop55_commit']=='4a910305b2cb74a4fc2b2c34baf44eb0542ff03f'
print('Phase29 verified; exact synthetic bundle SHA:',sha256(paths['bundle']))
print('Output folder:',OUT_DIR)

Phase29 verified; exact synthetic bundle SHA: 306f86faa010edf835629b2d911fc747fc19efaea8b4ce10dd219d7b97ff2a97
Output folder: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs


### Reconstruct source identity without trusting staging row order
The primary reconciliation key is a synthetic FHIR Observation reference, not a bare integer row number. The canonical numeric IDs are regenerated from sorted synthetic source identifiers **only to compare with the earlier Phase 27/29 staging**. This eliminates accidental row-order matches or swaps. The synthetic series fixture contains one series per ImagingStudy; this assumption does **not** carry to real ADNI studies.

In [3]:
# 3 — Canonical source manifest reconstructed independently from the synthetic FHIR bundle.
resources=[e['resource'] for e in bundle['entry']]
refs=[f"{r['resourceType']}/{r['id']}" for r in resources]
assert len(refs)==len(set(refs)),'Duplicate resource type/id in synthetic FHIR bundle'
idx={f"{r['resourceType']}/{r['id']}":r for r in resources}
patients=sorted((r for r in resources if r['resourceType']=='Patient'),key=lambda r:r['id'])
studies=sorted((r for r in resources if r['resourceType']=='ImagingStudy'),key=lambda r:r['id'])
obs=sorted((r for r in resources if r['resourceType']=='Observation'),key=lambda r:r['id'])
procedures=sorted((r for r in resources if r['resourceType']=='Procedure'),key=lambda r:r['id'])
assert (len(patients),len(studies),len(obs),len(procedures))==(2,4,16,4)
patient_id={p['id']:i for i,p in enumerate(patients,1)}
study_id={s['id']:i for i,s in enumerate(studies,1)}

def extension(resource,tail,value_field):
    found=[e.get(value_field) for e in resource.get('extension',[]) if e.get('url','').endswith('/'+tail)]
    return found[0] if len(found)==1 else None

def anatomy_side(o):
    coding=o.get('bodySite',{}).get('coding',[{}])[0].get('code','')
    m=re.fullmatch(r'SYN-(HIP|ENT)-(LEFT|RIGHT)',coding)
    return (m.group(1),m.group(2)) if m else (None,None)

def numeric_uid(uid):
    value=(uid or '').removeprefix('urn:oid:')
    return bool(re.fullmatch(r'[0-9]+(?:\.[0-9]+)*',value)) and len(value)<=64 and all(
        part=='0' or not part.startswith('0') for part in value.split('.'))

canonical=[]
for i,o in enumerate(obs,1):
    patient_ref=o.get('subject',{}).get('reference','')
    foci=[x.get('reference') for x in o.get('focus',[])]
    assert len(foci)==1 and foci[0] in idx and foci[0].startswith('ImagingStudy/')
    study=idx[foci[0]]
    assert study.get('subject',{}).get('reference')==patient_ref
    pid=patient_ref.split('/')[-1];sid=study['id']
    region,side=anatomy_side(o)
    assert region and side
    proc_ref=study.get('procedureReference',{}).get('reference')
    assert proc_ref in idx and proc_ref.startswith('Procedure/')
    procedure=idx[proc_ref]
    assert procedure.get('subject',{}).get('reference')==patient_ref
    assert procedure.get('performedDateTime')==study.get('started')
    dev_ref=o.get('device',{}).get('reference')
    device=idx[dev_ref]
    model_version=device.get('version',[{}])[0].get('value')
    provenance=[r for r in resources if r.get('resourceType')=='Provenance' and
                any(t.get('reference')==f"Observation/{o['id']}" for t in r.get('target',[]))]
    assert len(provenance)==1
    links={x.get('what',{}).get('reference') for x in provenance[0].get('entity',[])}
    assert foci[0] in links and dev_ref in links
    study_uids=[x.get('value') for x in study.get('identifier',[]) if x.get('system')=='urn:dicom:uid']
    assert len(study_uids)==1 and numeric_uid(study_uids[0])
    assert len(study.get('series',[]))==1
    series_uid=study['series'][0].get('uid')
    assert numeric_uid(series_uid)
    qty=o.get('valueQuantity',{})
    assert qty.get('system')=='http://unitsofmeasure.org' and qty.get('unit')=='mm3'
    assert isinstance(qty.get('value'),(int,float))
    canonical.append({
        'source_observation_id':o['id'], 'synthetic_record_key_sha256':hashlib.sha256(
            ('ICHI2027|'+pid+'|'+sid+'|'+o['id']).encode('utf-8')).hexdigest(),
        'source_study_id':sid,'synthetic_fhir_procedure_id':proc_ref.split('/')[-1],
        'synthetic_procedure_occurrence_id':study_id[sid],
        'image_occurrence_id':study_id[sid],
        'image_feature_id':i,'measurement_id':i,'synthetic_patient_key':pid,
        'synthetic_person_id':patient_id[pid], 'anatomy':region,'laterality':side,
        'baseline_or_followup':extension(o,'timepoint-label','valueString'),
        'value':qty['value'],'unit':qty['unit'],
        'algorithm_version':model_version,'trace_state':extension(o,'trace-state','valueCode'),
        'trace_rule':extension(o,'trace-rule-version','valueString'),
        'synthetic_study_uid':study_uids[0],'synthetic_series_uid':series_uid,
        'observation_provenance_count':len(provenance)
    })
canonical_df=pd.DataFrame(canonical)
assert canonical_df.synthetic_record_key_sha256.is_unique
assert canonical_df.synthetic_study_uid.nunique()==4
assert canonical_df.synthetic_series_uid.nunique()==4
print('Independently reconstructed',len(canonical_df),'synthetic evidence objects')
display(canonical_df[['source_observation_id','image_feature_id','image_occurrence_id','anatomy','laterality','baseline_or_followup']].head())

Independently reconstructed 16 synthetic evidence objects


,source_observation_id,image_feature_id,image_occurrence_id,anatomy,laterality,baseline_or_followup
0,obs-syn-p01-entorhinal-left-bl,1,1,ENT,LEFT,baseline
1,obs-syn-p01-entorhinal-left-m12,2,2,ENT,LEFT,12-month
2,obs-syn-p01-entorhinal-right-bl,3,1,ENT,RIGHT,baseline
3,obs-syn-p01-entorhinal-right-m12,4,2,ENT,RIGHT,12-month
4,obs-syn-p01-hippocampus-left-bl,5,1,HIP,LEFT,baseline


In [4]:
# 4 — Strict deterministic lineage validator + positive and negative controls.
FIELDS=[
    'source_study_id','synthetic_fhir_procedure_id','synthetic_procedure_occurrence_id',
    'image_occurrence_id','image_feature_id','measurement_id','synthetic_patient_key',
    'anatomy','laterality','baseline_or_followup','value','unit','algorithm_version',
    'trace_state','trace_rule'
]

def integrity_issues(actual,expected=canonical_df):
    errors=[]
    if actual.source_observation_id.duplicated().any():errors.append('DUPLICATE_SOURCE_ID')
    if set(actual.source_observation_id)!=set(expected.source_observation_id) or len(actual)!=len(expected):
        errors.append('SOURCE_SET_MISMATCH')
    exp=expected.set_index('source_observation_id',drop=False)
    for r in actual.to_dict('records'):
        key=r['source_observation_id']
        if key not in exp.index:continue
        correct=exp.loc[key]
        for f in FIELDS:
            actual_val=r.get(f)
            expected_val=correct[f]
            same=(abs(float(actual_val)-float(expected_val))<1e-8 if f=='value' and
                  actual_val not in ('',None) else str(actual_val)==str(expected_val))
            if not same:
                code={
                    'source_study_id':'STUDY_LINK_MISMATCH',
                    'synthetic_patient_key':'PATIENT_LINK_MISMATCH',
                    'image_feature_id':'FEATURE_ID_MISMATCH',
                    'measurement_id':'MEASUREMENT_ID_MISMATCH',
                    'image_occurrence_id':'OCCURRENCE_ID_MISMATCH',
                    'synthetic_fhir_procedure_id':'PROCEDURE_LINK_MISMATCH',
                    'baseline_or_followup':'TIMEPOINT_MISMATCH','value':'VALUE_MISMATCH',
                    'unit':'UNIT_MISMATCH','anatomy':'ANATOMY_MISMATCH',
                    'laterality':'LATERALITY_MISMATCH','trace_state':'TRACE_STATE_MISMATCH',
                    'trace_rule':'TRACE_RULE_MISMATCH','algorithm_version':'MODEL_VERSION_MISMATCH'
                }.get(f,'OTHER_LINEAGE_MISMATCH')
                errors.append(code)
    return sorted(set(errors))

clean_errors=integrity_issues(lineage29)
assert not clean_errors,'Phase29 sidecar fails independent source reconciliation: '+str(clean_errors)
mutations={}
mutations['E0_clean']=(lineage29.copy(deep=True),set())

m=lineage29.copy(deep=True);m=pd.concat([m,m.iloc[[0]]],ignore_index=True)
mutations['E1_duplicate_source']=(m,{'DUPLICATE_SOURCE_ID'})

m=lineage29.copy(deep=True)
other=canonical_df.loc[canonical_df.synthetic_patient_key!=m.iloc[0].synthetic_patient_key,'source_study_id'].iloc[0]
m.loc[0,'source_study_id']=other
mutations['E2_cross_patient_study']=(m,{'STUDY_LINK_MISMATCH'})

m=lineage29.copy(deep=True);m.loc[0,'baseline_or_followup']='WRONG_VISIT'
mutations['E3_timepoint_swap']=(m,{'TIMEPOINT_MISMATCH'})

m=lineage29.copy(deep=True);m.loc[0,'value']=float(m.loc[0,'value'])+100
mutations['E4_volume_corruption']=(m,{'VALUE_MISMATCH'})

m=lineage29.copy(deep=True);m.loc[0,'laterality']='RIGHT' if m.loc[0,'laterality']=='LEFT' else 'LEFT'
mutations['E5_laterality_corruption']=(m,{'LATERALITY_MISMATCH'})

m=lineage29.copy(deep=True);m.loc[0,'trace_state']=''
mutations['E6_trace_loss']=(m,{'TRACE_STATE_MISMATCH'})

m=lineage29.copy(deep=True);m.loc[0,'image_feature_id']=999
mutations['E7_wrong_feature_id']=(m,{'FEATURE_ID_MISMATCH'})

m=lineage29.copy(deep=True);m.loc[0,'synthetic_fhir_procedure_id']='proc-not-present'
mutations['E8_procedure_link_loss']=(m,{'PROCEDURE_LINK_MISMATCH'})

test_rows=[]
for name,(sample,want) in mutations.items():
    got=set(integrity_issues(sample))
    passed=(got==set() if not want else want.issubset(got))
    test_rows.append({'test':name,'passed':bool(passed),'expected_issues':';'.join(sorted(want)),
                      'observed_issues':';'.join(sorted(got))})
tests=pd.DataFrame(test_rows)
assert tests.passed.all(),tests.loc[~tests.passed].to_dict('records')
print(f'PASS: {tests.passed.sum()}/{len(tests)} synthetic lineage checks, including 8 negative controls')

PASS: 9/9 synthetic lineage checks, including 8 negative controls


In [5]:
# 5 — Independently check the original Phase27 row alignment when its staging artifacts exist.
# Missing original artifacts are NOT reported as a verified Phase27 comparison.
P27_FILES={
  'sidecar':'ICHI2027_Phase27_ARM_A_SYNTHETIC_PROVENANCE_SIDECAR*.csv',
  'image_feature':'ICHI2027_Phase27_ARM_A_IMAGE_FEATURE_STAGING*.csv',
  'measurement':'ICHI2027_Phase27_ARM_A_MEASUREMENT_STAGING*.csv',
}
p27_paths={k:sibling(OUT_DIR,pat) for k,pat in P27_FILES.items()}
p27_rows=[]
phase27_status='NOT_ATTESTED_MISSING_ORIGINAL_ARTIFACTS'
if all(p27_paths.values()):
    old=pd.read_csv(p27_paths['sidecar'],keep_default_na=False)
    oldfeat=pd.read_csv(p27_paths['image_feature'],keep_default_na=False)
    oldmeas=pd.read_csv(p27_paths['measurement'],keep_default_na=False)
    expected=canonical_df.set_index('source_observation_id')
    if len(old)==16 and old.synthetic_observation_id.is_unique and set(old.synthetic_observation_id)==set(expected.index):
        for _,r in old.iterrows():
            e=expected.loc[r.synthetic_observation_id]
            checks={
              'study':str(r.source_study_id)==str(e.source_study_id),
              'feature_id':int(r.staging_image_feature_id)==int(e.image_feature_id),
              'measurement_id':int(r.staging_measurement_id)==int(e.measurement_id),
              'unit':str(r.original_unit)==str(e.unit),
              'trace_state':str(r.synthetic_trace_state)==str(e.trace_state),
              'trace_rule':str(r.synthetic_trace_rule_version)==str(e.trace_rule),
              'region_side':str(r.synthetic_anatomy_laterality)==
                 ('SYN-'+str(e.anatomy)+'-'+str(e.laterality)),
              'model_version':str(r.source_model_version)==str(e.algorithm_version)
            }
            p27_rows.append({'source_observation_id':r.synthetic_observation_id,
                             'phase27_feature_id':r.staging_image_feature_id,
                             'phase30_canonical_feature_id':int(e.image_feature_id),
                             'passed':all(checks.values()),
                             'failed_dimensions':';'.join(k for k,v in checks.items() if not v)})
        p27_feat_ids=set(pd.to_numeric(oldfeat.image_feature_id,errors='coerce').dropna().astype(int))
        p27_meas_ids=set(pd.to_numeric(oldmeas.measurement_id,errors='coerce').dropna().astype(int))
        if (len(p27_rows)==16 and all(x['passed'] for x in p27_rows) and
            p27_feat_ids==set(range(1,17)) and p27_meas_ids==set(range(1,17))):
            phase27_status='VERIFIED_FROM_ORIGINAL_PHASE27_STAGING_AND_SIDECAR'
        else:
            phase27_status='FAILED_ORIGINAL_PHASE27_COMPARISON'
    else:
        phase27_status='FAILED_ORIGINAL_PHASE27_SOURCE_SET'
else:
    print('Phase27 original stage/sidecar absent: independent Phase27 comparison NOT ATTESTED.')
phase27_check=pd.DataFrame(p27_rows,columns=[
    'source_observation_id','phase27_feature_id','phase30_canonical_feature_id',
    'passed','failed_dimensions'])
print('Original Phase27 ordering:',phase27_status)
assert not phase27_status.startswith('FAILED'), 'Original Phase27 mapping conflicts with canonical source; investigate before proceeding.'

Original Phase27 ordering: VERIFIED_FROM_ORIGINAL_PHASE27_STAGING_AND_SIDECAR


### Private Athena intake and human adjudication
The **19 binding keys** originate in Phase 29. This notebook does not guess their concept IDs from names. It generates a review template, optionally suggests **unapproved search candidates privately** when a frozen local vocabulary exists, and validates only explicitly reviewed IDs against that exact snapshot. For an unpinned metadata/event-field definition, a reviewer must document the target table/column and its definition; otherwise the event link remains in the sidecar.

Use your official, licensed Athena export: `CONCEPT.csv` or `CONCEPT.csv.gz` inside the same Drive folder's `_PRIVATE_ATHENA` directory. **Do not upload the raw vocabulary or private candidate file to public GitHub.**

In [6]:
# 6 — Human-review template, never overwriting an existing review.
EXPECTED_DOMAINS={
    'PERSON_GENDER_F':'Gender','PERSON_GENDER_M':'Gender',
    'PERSON_RACE_UNRECORDED':'Race','PERSON_ETHNICITY_UNRECORDED':'Ethnicity',
    'PROCEDURE_MRI':'Procedure','PROCEDURE_TYPE_SYN':'Type Concept',
    'MEAS_HIP_LEFT':'Measurement','MEAS_HIP_RIGHT':'Measurement',
    'MEAS_ENT_LEFT':'Measurement','MEAS_ENT_RIGHT':'Measurement',
    'MEAS_TYPE_SYN':'Type Concept','UNIT_MM3':'Unit',
    'IF_EVENT_MEAS_ID':'Metadata'
}
SEARCH_TERMS={
    'IO_MODALITY_MR':'magnetic resonance',
    'IF_HIP_LEFT':'hippocamp', 'IF_HIP_RIGHT':'hippocamp',
    'IF_ENT_LEFT':'entorhin','IF_ENT_RIGHT':'entorhin',
    'IF_TYPE_ALGORITHM':'algorithm',
    'PERSON_GENDER_F':'female','PERSON_GENDER_M':'male',
    'PERSON_RACE_UNRECORDED':'unknown',
    'PERSON_ETHNICITY_UNRECORDED':'unknown',
    'PROCEDURE_MRI':'brain magnetic resonance',
    'PROCEDURE_TYPE_SYN':'derived',
    'MEAS_HIP_LEFT':'hippocamp','MEAS_HIP_RIGHT':'hippocamp',
    'MEAS_ENT_LEFT':'entorhin','MEAS_ENT_RIGHT':'entorhin',
    'MEAS_TYPE_SYN':'derived', 'UNIT_MM3':'cubic millimeter',
    'IF_EVENT_MEAS_ID':'measurement_id'
}
assert set(SEARCH_TERMS)==set(bind29.binding_key)
TEMPLATE_PATH=OUT_DIR/'ICHI2027_Phase30_BINDING_REVIEW_TEMPLATE.csv'
REVIEW_PATH=OUT_DIR/'ICHI2027_Phase30_REVIEWED_BINDINGS.csv'
template=pd.DataFrame([{
    'binding_key':str(r.binding_key),'target_field':str(r.target_field),
    'synthetic_source_code':str(r.synthetic_source_code),
    'search_hint_NOT_A_MAPPING':SEARCH_TERMS[str(r.binding_key)],
    'expected_domain_if_known':EXPECTED_DOMAINS.get(str(r.binding_key),''),
    'concept_id':'', 'reviewed_domain_id':'', 'reviewer':'', 'review_date':'',
    'mapping_evidence':'', 'event_field_cdm_table_field':'',
    'event_field_definition_evidence':''
    } for r in bind29.itertuples(index=False)])
if not TEMPLATE_PATH.is_file():template.to_csv(TEMPLATE_PATH,index=False)
if REVIEW_PATH.is_file():
    reviewed=pd.read_csv(REVIEW_PATH,dtype=str,keep_default_na=False)
    required_review_cols={'binding_key','concept_id','reviewed_domain_id','reviewer','review_date','mapping_evidence',
                          'event_field_cdm_table_field','event_field_definition_evidence'}
    if not required_review_cols.issubset(reviewed.columns):
        raise ValueError('Reviewed bindings CSV missing: '+str(sorted(required_review_cols-set(reviewed.columns))))
    if reviewed.binding_key.duplicated().any():raise ValueError('Duplicate reviewed binding keys')
    if not set(reviewed.binding_key).issubset(set(template.binding_key)):
        raise ValueError('Unrecognized binding key supplied')
    review_map={r['binding_key']:r for r in reviewed.to_dict('records')}
else:
    review_map={}
PRIVATE=OUT_DIR/'_PRIVATE_ATHENA'
CONCEPT=next((PRIVATE/n for n in ['CONCEPT.csv','concept.csv','CONCEPT.csv.gz','concept.csv.gz']
              if (PRIVATE/n).is_file()),None)
print('Review template:',TEMPLATE_PATH.name)
print('Review-file rows (including any blank entries):',len(review_map),'of 19; private Athena snapshot:',bool(CONCEPT))

Review template: ICHI2027_Phase30_BINDING_REVIEW_TEMPLATE.csv
Review-file rows (including any blank entries): 0 of 19; private Athena snapshot: False


In [7]:
# 7 — Validate reviewed concepts ONLY against the user's private frozen snapshot.
# Optionally create private search hints, NOT approved mappings. Raw CONCEPT.csv never leaves _PRIVATE_ATHENA.
requested=set()
for k,r in review_map.items():
    c=str(r.get('concept_id','')).strip()
    if c:
        if not c.isdigit() or int(c)<=0:raise ValueError('Reviewed concept_id must be a positive integer for '+k)
        requested.add(c)

snapshot_hash=sha256(CONCEPT) if CONCEPT else None
matched={};candidate_rows=[]
if CONCEPT:
    PRIVATE.mkdir(parents=True,exist_ok=True)
    file_open=gzip.open if CONCEPT.suffix=='.gz' else open
    with file_open(CONCEPT,'rt',encoding='utf-8-sig',errors='replace') as f:
        first=f.readline()
    sep='\t' if '\t' in first else ','
    wanted=['concept_id','concept_name','domain_id','standard_concept',
            'valid_start_date','valid_end_date','invalid_reason']
    with file_open(CONCEPT,'rt',encoding='utf-8-sig',errors='replace') as f:
        header=f.readline().strip().split(sep)
    if not set(wanted).issubset(set(header)):
        raise RuntimeError('Private CONCEPT.csv missing official Athena fields: '+str(sorted(set(wanted)-set(header))))
    hints=sorted(set(SEARCH_TERMS.values()))
    quotas={h:0 for h in hints}
    for chunk in pd.read_csv(CONCEPT,sep=sep,compression='infer',dtype=str,
                             keep_default_na=False,chunksize=50000,usecols=wanted):
        for item in chunk.loc[chunk.concept_id.isin(requested)].to_dict('records'):
            matched[item['concept_id']]=item
        # Private, non-binding candidate list; maximum six valid, standard candidates per search term.
        valid=chunk.loc[(chunk.invalid_reason=='') & (chunk.standard_concept=='S')]
        for term in hints:
            if quotas[term]>=6:continue
            pool=chunk.loc[(chunk.invalid_reason=='') & (chunk.domain_id=='Metadata')] if term=='measurement_id' else valid
            subset=pool.loc[pool.concept_name.str.contains(term,case=False,regex=False,na=False)]
            for cr in subset.head(6-quotas[term]).to_dict('records'):
                candidate_rows.append({'search_hint':term,**cr,'status':'SEARCH_HINT_ONLY_REVIEW_REQUIRED'})
                quotas[term]+=1
    private_candidate_path=PRIVATE/'ICHI2027_Phase30_PRIVATE_CONCEPT_CANDIDATES.csv'
    pd.DataFrame(candidate_rows,columns=['search_hint']+wanted+['status']).to_csv(private_candidate_path,index=False)
else:
    private_candidate_path=None

snapshot_checked=[];approved_keys=set()
asof=date.today()
def parse_date(raw):
    raw=str(raw or '').strip()
    if not raw:return None
    for fmt in ['%Y%m%d','%Y-%m-%d']:
        try:return datetime.strptime(raw,fmt).date()
        except ValueError:pass
    return None

for row in template.to_dict('records'):
    k=row['binding_key'];r=review_map.get(k,{})
    cid=str(r.get('concept_id','')).strip()
    status='UNREVIEWED';snapshot_found=False
    if cid:
        if not all(str(r.get(f,'')).strip() for f in ['reviewer','review_date','mapping_evidence']):
            status='BLOCKED_INCOMPLETE_HUMAN_REVIEW'
        elif not parse_date(r.get('review_date')):
            status='BLOCKED_INVALID_REVIEW_DATE'
        elif not CONCEPT:
            status='BLOCKED_NO_FROZEN_CONCEPT_CSV'
        elif cid not in matched:
            status='CONCEPT_NOT_FOUND_IN_SNAPSHOT'
        else:
            snapshot_found=True;c=matched[cid]
            start=parse_date(c['valid_start_date']);end=parse_date(c['valid_end_date'])
            expdom=EXPECTED_DOMAINS.get(k,'')
            declared_dom=str(r.get('reviewed_domain_id','')).strip()
            if c['invalid_reason'] or not start or not end or not(start<=asof<=end):
                status='BLOCKED_INVALID_OR_EXPIRED_CONCEPT'
            elif expdom and c['domain_id']!=expdom:
                status='BLOCKED_WRONG_CONCEPT_DOMAIN'
            elif declared_dom!=c['domain_id']:
                status='BLOCKED_REVIEWED_DOMAIN_MISMATCH_OR_MISSING'
            elif k!='IF_EVENT_MEAS_ID' and c['standard_concept']!='S':
                status='BLOCKED_NOT_STANDARD_CONCEPT'
            elif k=='IF_EVENT_MEAS_ID' and not(
                    c['domain_id']=='Metadata' and
                    c['concept_name'].strip().lower()=='measurement_id' and
                    str(r.get('event_field_cdm_table_field','')).strip().upper()=='MEASUREMENT.MEASUREMENT_ID' and
                    str(r.get('event_field_definition_evidence','')).strip()):
                status='BLOCKED_METADATA_EVENT_FIELD_NEEDS_ADJUDICATION'
            else:
                status='REVIEWED_AND_VERIFIED_IN_FROZEN_SNAPSHOT_TEST_ONLY'
                approved_keys.add(k)
    snapshot_checked.append({
        'binding_key':k,'target_field':row['target_field'],
        'validation_status':status,'snapshot_checked':bool(snapshot_found),
        'review_submitted':bool(cid),'approved_for_synthetic_staging_next_phase':k in approved_keys
    })
binding_validation=pd.DataFrame(snapshot_checked)
print('Snapshot-validated + human-reviewed test bindings:',len(approved_keys),'of 19')
if not CONCEPT:print('No CONCEPT.csv supplied: no concept IDs have been asserted or invented.')

Snapshot-validated + human-reviewed test bindings: 0 of 19
No CONCEPT.csv supplied: no concept IDs have been asserted or invented.


In [8]:
# 8 — Reconcile all original 48 gap instances without silently treating a suggested concept as resolved.
# Prepare a precise handoff for the NEXT phase: reviewed concepts must still be applied to target staging.
region_key={('HIP','LEFT'):'IF_HIP_LEFT',('HIP','RIGHT'):'IF_HIP_RIGHT',
            ('ENT','LEFT'):'IF_ENT_LEFT',('ENT','RIGHT'):'IF_ENT_RIGHT'}
by_feat=canonical_df.set_index('image_feature_id')
disposition=gaps29.copy(deep=True)
next_bindings=[];next_status=[]
for row in disposition.to_dict('records'):
    target=row['table']+'.'+row['required_field']
    key=''
    if target=='image_occurrence.modality_concept_id':key='IO_MODALITY_MR'
    elif target=='image_feature.image_feature_type_concept_id':key='IF_TYPE_ALGORITHM'
    elif target=='image_feature.image_feature_concept_id':
        feature_id=int(row['staging_row_id'])
        if feature_id not in by_feat.index:raise AssertionError('Original Phase29 gap refers to unknown feature id')
        f=by_feat.loc[feature_id]
        key=region_key[(f['anatomy'],f['laterality'])]
    next_bindings.append(key)
    if row['resolution_status']!='BLOCKED_VOCABULARY_VALIDATION':
        next_status.append(row['resolution_status'])
    elif key in approved_keys:
        next_status.append('REVIEWED_BINDING_AVAILABLE_NOT_YET_APPLIED_TO_STAGE')
    else:
        next_status.append('BLOCKED_VOCABULARY_VALIDATION')
disposition['phase30_binding_key']=next_bindings
disposition['phase30_status']=next_status
assert len(disposition)==48
assert disposition.groupby('phase30_status').size().sum()==48
still_blocked=int((disposition.phase30_status=='BLOCKED_VOCABULARY_VALIDATION').sum())
ready_for_application=int((disposition.phase30_status=='REVIEWED_BINDING_AVAILABLE_NOT_YET_APPLIED_TO_STAGE').sum())
assert still_blocked+ready_for_application==36
print('Original 48 gap instances: 8 synthetic UIDs; 4 synthetic procedures; ',
      still_blocked,'still blocked; ',ready_for_application,'have reviewed bindings ready for next-phase staging.')
print('Previously reported 82 mandatory target-field instances remain a Phase29 staging observation; '
      'this notebook does not falsely mark target fields filled.')

Original 48 gap instances: 8 synthetic UIDs; 4 synthetic procedures;  36 still blocked;  0 have reviewed bindings ready for next-phase staging.
Previously reported 82 mandatory target-field instances remain a Phase29 staging observation; this notebook does not falsely mark target fields filled.


In [9]:
# 9 — Save only synthetic/public-safe outputs alongside the Phase29 files, then read back.
OUTPUTS={
 'shareable':OUT_DIR/'ICHI2027_Phase30_SHAREABLE.json',
 'canonical_lineage':OUT_DIR/'ICHI2027_Phase30_CANONICAL_SYNTHETIC_LINEAGE.csv',
 'integrity_tests':OUT_DIR/'ICHI2027_Phase30_LINEAGE_INTEGRITY_TESTS.csv',
 'phase27_comparison':OUT_DIR/'ICHI2027_Phase30_PHASE27_ROW_COMPARISON.csv',
 'binding_status':OUT_DIR/'ICHI2027_Phase30_BINDING_VALIDATION_STATUS.csv',
 'gap_disposition':OUT_DIR/'ICHI2027_Phase30_GAP_DISPOSITION.csv',
}
canonical_df.to_csv(OUTPUTS['canonical_lineage'],index=False)
tests.to_csv(OUTPUTS['integrity_tests'],index=False)
phase27_check.to_csv(OUTPUTS['phase27_comparison'],index=False)
binding_validation.to_csv(OUTPUTS['binding_status'],index=False)
disposition.to_csv(OUTPUTS['gap_disposition'],index=False)

summary={
 'project':'ICHI2027_neuroimaging_fhir_omop_fidelity',
 'phase':'30_deterministic_synthetic_lineage_and_private_athena_binding_gate',
 'run_utc':datetime.now(timezone.utc).isoformat(),
 'privacy':'Public outputs contain synthetic identifiers and status only; raw/private Athena vocabulary and candidate rows remain private; no real ADNI data.',
 'phase29_sha256':sha256(P29_JSON),
 'phase29_bundle_sha256':sha256(paths['bundle']),
 'pinned_imagewg_commit':p29['pinned_imagewg_commit'],
 'pinned_omop55_commit':p29['pinned_omop55_commit'],
 'synthetic_lineage':{
    'fhir_observations':len(obs),'source_to_phase29_target_keys_verified':True,
    'synthetic_persons':len(patients),'synthetic_imaging_studies':len(studies),
    'synthetic_procedures':len(procedures),
    'synthetic_integrity_test_count':len(tests),
    'synthetic_integrity_tests_passing':int(tests.passed.sum()),
    'original_phase27_row_order_status':phase27_status,
    'original_phase27_row_order_independently_verified':phase27_status.startswith('VERIFIED'),
    'real_imaging_procedure_or_UID_provenance_attested':False,
 },
 'vocabulary':{
    'local_frozen_snapshot_provided':bool(CONCEPT),
    'private_snapshot_sha256':snapshot_hash,
    'reviewed_file_supplied':bool(REVIEW_PATH.is_file()),
    'reviewed_snapshot_verified_test_binding_count':len(approved_keys),
    'total_binding_keys':len(template),
    'private_candidate_file_created':bool(private_candidate_path),
    'concepts_automatically_selected':0,
    'Athena_mapping_approval_claimed':False,
 },
 'original_gap_tracking':{
    'original_required_instances':48,
    'synthetic_uid_only_closed':8,'manufactured_synthetic_procedures_only_closed':4,
    'reviewed_bindings_ready_but_not_applied':ready_for_application,
    'remaining_vocabulary_blocked':still_blocked,
    'phase29_required_target_missing_instances_unmodified':int(p29['target_required_missing_instances']),
 },
 'gates':{
    'phase29_verified':True,
    'deterministic_synthetic_lineage_verified':True,
    'original_phase27_row_order_independently_verified':phase27_status.startswith('VERIFIED'),
    'frozen_vocabulary_snapshot_provided':bool(CONCEPT),
    'all_19_bindings_reviewed_in_snapshot':len(approved_keys)==19,
    'all_36_original_vocab_gaps_have_reviewed_binding_ready':ready_for_application==36,
    'synthetic_armA_faithful_target_mapping_completed':False,
    'OMOP_target_cohort_equivalence_tested':False,
    'pyOMOP_ETL_executed':False,
    'official_MICDM_load_completed':False,
    'real_MRI_rows_processed':0,
    'true_four_arm_comparison_completed':False,
    'original_generic_armC_executed':False,
 },
 'next_stage_gate':(
    'Supply private frozen Athena CONCEPT.csv plus 19 independently reviewed binding decisions; '
    'reconcile optional original Phase27 staging if available; independently validate event-field metadata. '
    'Only then apply approved bindings to rebuilt synthetic Arm A staging and run exact required-field, '
    'cardinality and idempotence checks. No real-data or four-arm inference.'
 )
}
OUTPUTS['shareable'].write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding='utf-8')
readback=json.loads(OUTPUTS['shareable'].read_text(encoding='utf-8'))
assert readback['synthetic_lineage']['synthetic_integrity_tests_passing']==9
assert sha256(OUTPUTS['canonical_lineage'])
assert len(pd.read_csv(OUTPUTS['gap_disposition']))==48
assert readback['gates']['synthetic_armA_faithful_target_mapping_completed'] is False
print('PHASE 30 SYNTHETIC LINEAGE: PASS')
print('Original Phase27 order check:',phase27_status)
print('Snapshot and review:',bool(CONCEPT),'/',len(approved_keys),'validated review bindings')
print('Saved directly to Google Drive:')
for p in OUTPUTS.values():print(' -',p.name)
print(' -',TEMPLATE_PATH.name)
if private_candidate_path:print('Private only:',private_candidate_path.name)

PHASE 30 SYNTHETIC LINEAGE: PASS
Original Phase27 order check: VERIFIED_FROM_ORIGINAL_PHASE27_STAGING_AND_SIDECAR
Snapshot and review: False / 0 validated review bindings
Saved directly to Google Drive:
 - ICHI2027_Phase30_SHAREABLE.json
 - ICHI2027_Phase30_CANONICAL_SYNTHETIC_LINEAGE.csv
 - ICHI2027_Phase30_LINEAGE_INTEGRITY_TESTS.csv
 - ICHI2027_Phase30_PHASE27_ROW_COMPARISON.csv
 - ICHI2027_Phase30_BINDING_VALIDATION_STATUS.csv
 - ICHI2027_Phase30_GAP_DISPOSITION.csv
 - ICHI2027_Phase30_BINDING_REVIEW_TEMPLATE.csv


### What to return for Phase 31
Upload `ICHI2027_Phase30_SHAREABLE.json`, `ICHI2027_Phase30_LINEAGE_INTEGRITY_TESTS.csv`, and `ICHI2027_Phase30_BINDING_VALIDATION_STATUS.csv`. If a snapshot and manual review are available, also include the **public-safe binding status** file, **not** the private `CONCEPT.csv`, private candidates, or an unreviewed concept-ID guess. Phase 31 can then apply actual verified mappings to a new *synthetic* staging rebuild and test required-field completion, linkage and idempotence; absent vocabulary, the 36 original vocabulary gaps remain documented blockers.